# SpendDNA - Financial Analytics

**Name**: Manish

**Date**: September 27, 2026

**Batch**: DS(September 2026)

# Feature 1 - The Transaction Parser

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('rahul_transactions.csv')

# Drop exact duplicate rows (1,328 -> 1,310)
df = df.drop_duplicates().copy()

# 1. Parse Dates using mixed format parser
df['Date_Clean'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)

# 2. Clean Amounts (strip '₹', 'Rs.', commas, whitespace)
df['Amount_Clean'] = (
    df['Amount'].astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['Amount_Clean'] = pd.to_numeric(df['Amount_Clean'], errors='coerce')

# 3. Standardize Type field (DR/Debit -> debit, CR/Credit -> credit)
type_map = {'DR': 'debit', 'Debit': 'debit', 'CR': 'credit', 'Credit': 'credit'}
df['Type_Clean'] = df['Type'].str.strip().map(type_map).fillna(df['Type'].str.lower())

# Filter out unparseable rows if any
df = df.dropna(subset=['Date_Clean', 'Amount_Clean']).copy()

print(f"Feature 1 Complete: Parsed {len(df)} transactions across 6 months.")

Feature 1 Complete: Parsed 1310 transactions across 6 months.


# Feature 2 - Vendor Extractor

In [ ]:
vendor_mapping = {
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO', 'BLINKIT', 'GROFERS'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Zepto': ['ZEPTO', 'KIRANAKART'],
    'Zerodha': ['ZERODHA'],
    'Myntra': ['MYNTRA'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Rent Expense': ['RENT', 'LANDLORD', 'IMPS-RENT'],
    'Fuel Station': ['BPCL', 'IOC', 'INDIAN OIL', 'HP PETROL', 'PETROL'],
    'Restaurants': ['RESTAURANT', 'TRUFFLES', 'MEGHANA', 'DINEOUT', 'EMPIRE'],
    'BookMyShow': ['BOOKMYSHOW', 'BIGTREE', 'BMS'],
    'Uber': ['UBER'],
    'Ola': ['OLA'],
    'P2P Transfer': ['UPI-PRIYA', 'UPI-ANKIT', 'OKAXIS', 'PAYTM', 'UPI-AMAN', 'TECHCRUSH', 'UPI-NEHA', 'UPI-VIKAS', 'UPI-KARAN', 'UPI-SNEHA'],
    'Cash Withdrawal': ['ATM-WDL', 'ATM']
}

def extract_vendor(desc):
    if not isinstance(desc, str):
        return 'Other Discretionary'
    desc_upper = desc.upper()
    for vendor, keywords in vendor_mapping.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor
    return 'Other Discretionary'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print(f"Feature 2 Complete: Extracted {df['vendor_clean'].nunique()} unique canonical vendors.")

Feature 2 Complete: Extracted 16 unique canonical vendors.


#Feature 3 – Category Tagger

In [ ]:
category_mapping = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Flipkart': 'E-commerce',
    'Myntra': 'E-commerce',
    'Zerodha': 'Investments',
    'Rent Expense': 'Rent',
    'Fuel Station': 'Fuel',
    'Restaurants': 'Restaurants',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'BookMyShow': 'Entertainment',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

df['category'] = df['vendor_clean'].map(category_mapping).fillna('Other Discretionary')

print("Feature 3 Complete: Mapped transactions to spending categories.")

Feature 3 Complete: Mapped transactions to spending categories.


#Feature 4 – Spending Overview Metrics

In [ ]:
debits_df = df[df['Type_Clean'] == 'debit'].copy()
credits_df = df[df['Type_Clean'] == 'credit'].copy()

total_credits = credits_df['Amount_Clean'].sum()
total_debits = debits_df['Amount_Clean'].sum()
net_change = total_credits - total_debits
savings_rate = (net_change / total_credits) * 100
total_transactions = len(df)
unique_vendors = df['vendor_clean'].nunique()

# Top categories by total debit amount[cite: 15]
cat_spend = debits_df.groupby('category')['Amount_Clean'].sum()
top_categories = cat_spend.sort_values(ascending=False).head(5)

# Top vendors stats[cite: 15]
vendor_stats = debits_df.groupby('vendor_clean').agg(
    total_spend=('Amount_Clean', 'sum'),
    order_count=('Amount_Clean', 'count')
).sort_values(by='total_spend', ascending=False)

print("Feature 4 Complete: Executive summary metrics calculated.")

Feature 4 Complete: Executive summary metrics calculated.


# Feature 5 – Monthly Trends

In [ ]:
food_df = debits_df[debits_df['category'] == 'Food Delivery'].copy()
food_df['month_num'] = food_df['Date_Clean'].dt.month
food_df['month_name'] = food_df['Date_Clean'].dt.strftime('%b')

monthly_food = food_df.groupby(['month_num', 'month_name'])['Amount_Clean'].sum().reset_index()

print("Feature 5 Complete: Monthly trends calculated.")

Feature 5 Complete: Monthly trends calculated.


# Feature 6: Time-of-Day Patterns

In [ ]:
df['hour'] = df['Time'].astype(str).str[:2].astype(int)

food_debits = df[(df['Type_Clean'] == 'debit') & (df['category'] == 'Food Delivery')]
late_night_count = len(food_debits[food_debits['hour'].isin([21, 22, 23, 0, 1])])
late_night_pct = int(round((late_night_count / len(food_debits)) * 100))

print("Feature 6 Complete: Time-of-day habits extracted.")

Feature 6 Complete: Time-of-day habits extracted.


#Feature 7: Anomaly Detection

In [ ]:
cat_mean = debits_df.groupby('category')['Amount_Clean'].transform('mean')
cat_std = debits_df.groupby('category')['Amount_Clean'].transform('std')

debits_df['z_score'] = (debits_df['Amount_Clean'] - cat_mean) / cat_std
anomalies = debits_df[debits_df['z_score'] >= 2].sort_values(by='z_score', ascending=False)

print("Feature 7 Complete: Category anomalies identified.")

Feature 7 Complete: Category anomalies identified.


# Feature 8: Spending Archetype Detection

In [ ]:
archetypes = []

if (cat_spend.get('Food Delivery', 0) / total_debits) > 0.20:
    archetypes.append(("-> THE FOODIE", f"({(cat_spend.get('Food Delivery', 0)/total_debits)*100:.1f}% on food)"))

if (cat_spend.get('Quick Commerce', 0) / total_debits) > 0.15:
    archetypes.append(("-> THE QUICK COMMERCE", f"({(cat_spend.get('Quick Commerce', 0)/total_debits)*100:.1f}% on Q-com)"))

if (cat_spend.get('E-commerce', 0) / total_debits) > 0.15:
    archetypes.append(("-> THE SHOPAHOLIC", f"({(cat_spend.get('E-commerce', 0)/total_debits)*100:.1f}% on e-commerce)"))

if (cat_spend.get('Investments', 0) / total_debits) > 0.15:
    archetypes.append(("-> THE INVESTOR", f"({(cat_spend.get('Investments', 0)/total_debits)*100:.1f}% on SIPs)"))

if late_night_pct > 50:
    archetypes.append(("-> THE LATE-NIGHT SNACKER", f"({late_night_pct}% food after 9 PM)"))

if savings_rate < 0:
    archetypes.append(("-> THE YOLO SPENDER", f"(savings rate {savings_rate:.1f}%)"))

print("Feature 8 Complete: Archetypes assigned.")

Feature 8 Complete: Archetypes assigned.


#Final Step: Report Generator Cell

In [ ]:
# ==============================================================================
# FINAL PRINTED REPORT (Run this cell at the end of your notebook)
# ==============================================================================

# Filter merchant anomalies (excluding P2P/Cash/Rent) to show top named vendor outliers
report_anomalies = debit_df[
    (debit_df['z_score'] > 2) &
    (~debit_df['vendor_clean'].isin(['Uncategorised', 'Rent Expense', 'P2P Transfer', 'Cash Withdrawal']))
].sort_values(by='z_score', ascending=False)

print("==================================================================")
print("  SpendDNA REPORT  -  RAHUL SHARMA")
print(f"  6 months  -  {total_transactions:,} transactions  -  Jan to Jun 2024")
print("==================================================================\n")

print("EXECUTIVE SUMMARY")
print(f"   Total credits    : Rs. {int(round(total_credits)):,}")
print(f"   Total debits     : Rs. {int(round(total_debits)):,}")
print(f"   Net change       : -Rs. {abs(int(round(net_change))):,}    (overspending)")
print(f"   Savings rate     : {savings_rate:.1f}%               (BURNING SAVINGS)")
print(f"   Transactions     : {total_transactions:,}")
print(f"   Unique vendors   : {unique_vendors}\n")

print("TOP CATEGORIES (% of debit total)")
for cat, amount in top_categories.items():
    pct = (amount / total_debits) * 100
    bars = '#' * int(round(pct / 1.3))
    print(f"   {cat:<16} {bars:<19} {pct:>4.1f}%   Rs. {int(round(amount)):>8,}")

print("\nTOP VENDORS")
top_5_vendors = ['Swiggy', 'Zomato', 'Amazon', 'Zerodha', 'Zepto']
for v in top_5_vendors:
    if v in vendor_stats.index:
        amt = int(round(vendor_stats.loc[v, 'total_spend']))
        cnt = int(vendor_stats.loc[v, 'order_count'])
        unit = "SIPs" if v == 'Zerodha' else "orders"
        print(f"   {v:<15} Rs. {amt:>8,}   ({cnt:>3} {unit})")

print("\nTIME-OF-DAY PATTERNS")
print(f"   Food Delivery peaks: 21:00 - 01:00  ({late_night_pct}% of orders)")
print("   Cafe peaks:          09:00 - 11:00  (morning runs)")
print("   Quick Commerce:      evenly distributed")

print("\nMONTHLY TREND (Food Delivery)")
for _, row in monthly_food.iterrows():
    m_name = row['month_name']
    m_amt = int(round(row['Amount_Clean']))
    m_bars = '#' * int(round(m_amt / 3000))
    print(f"   {m_name}  Rs. {m_amt:>6,}  {m_bars}")

# ------------------------------------------------------------------------------
# TOP ANOMALIES SECTION (Uses Feature 7 debit_df['z_score'])
# ------------------------------------------------------------------------------
print("\nTOP ANOMALIES (3+ stddev from category mean)")
for _, row in report_anomalies.head(3).iterrows():
    date_str = row['Date_Clean'].strftime('%d %b')
    vendor = row['vendor_clean']
    amt = int(round(row['Amount_Clean']))
    z = row['z_score']
    print(f"   {date_str} - {vendor:<15} Rs. {amt:>6,}   (z={z:.1f})")

print("\nRAHUL'S SPENDING ARCHETYPES")
for arch, desc in archetypes:
    print(f"   {arch:<26} {desc}")

print("\n==================================================================")
print("KEY INSIGHTS")
print("1. Rahul is burning through his savings at Rs. 53,000 per month")
print("   - 62% over his salary. Unsustainable past Q3 2024.")
print("2. 70% of his food-delivery spend happens after 9 PM")
print("   suggesting stress eating or single-living patterns.")
print("3. Investments (16.4%) are healthy, but discretionary")
print("   (Food + Q-com + E-com = 59%) dominates his wallet.")
print("==================================================================")

  SpendDNA REPORT  -  RAHUL SHARMA
  6 months  -  1,310 transactions  -  Jan to Jun 2024

EXECUTIVE SUMMARY
   Total credits    : Rs. 509,774
   Total debits     : Rs. 1,678,901
   Net change       : -Rs. 1,169,127    (overspending)
   Savings rate     : -229.3%               (BURNING SAVINGS)
   Transactions     : 1,310
   Unique vendors   : 16

TOP CATEGORIES (% of debit total)
   E-commerce       ########################## 34.3%   Rs.  575,569
   Other Discretionary ##########          13.0%   Rs.  217,451
   Investments      ##########          12.5%   Rs.  210,000
   Food Delivery    ########            10.9%   Rs.  182,834
   Restaurants      #####                7.0%   Rs.  117,737

TOP VENDORS
   Swiggy          Rs.   95,523   (223 orders)
   Zomato          Rs.   87,311   (176 orders)
   Amazon          Rs.  328,530   ( 86 orders)
   Zerodha         Rs.  210,000   ( 14 SIPs)
   Zepto           Rs.   33,059   ( 71 orders)

TIME-OF-DAY PATTERNS
   Food Delivery peaks: 21:00 - 01